## Live GPT request timing

For every selected BigBench Movie datapoint, this runs one vanilla sample followed by 100 step-bootstrap samples through `GptAdapter`. Step-bootstrap requests use bounded async concurrency (`API_CONCURRENCY`). This is generation-only (no confidence scoring): each datapoint makes 2 API calls for the two-phase vanilla generation and 100 API calls for step bootstrap. Running the benchmark incurs API cost.

In [3]:
print("hello world")
import sys
print(sys.executable)

hello world
/pm/miniconda3/envs/users/han/cot/bin/python3.10


In [7]:
import os
from time import perf_counter
from dotenv import load_dotenv
from tqdm.auto import tqdm

from config import GenerationConfig, SamplingConfig
from datasets.bigbench_movie import BigBenchMovieDataset
from domain import PromptRequest
from models.adapters.gpt_adapter import GptAdapter
from pipeline.sampling.context import SampleContext
from pipeline.sampling.stepbootstrap_sampling import StepBootstrapSampling
from pipeline.sampling.vanilla_sampling import VanillaSampling

N = 1
MAX_TOKENS = 1024
NB_STEPBOOTSTRAP_SAMPLES = 100
API_CONCURRENCY = 100
load_dotenv()
dataset = BigBenchMovieDataset()
datapoints = dataset.load_datapoints()[:N]

if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError("Set OPENAI_API_KEY before running this benchmark.")
adapter = GptAdapter()
generation_config = GenerationConfig(
    model="gpt", dataset="bigbench_movie", backend="api", prompt_type=2,
    max_tokens=MAX_TOKENS, sample_size=N, sample_range=None, sample_indices=None,
    from_pickle=None, from_pregenerated=None, discord=False, tag=None,
    debug_nocache=False, experimental_llama_batch=False, api_concurrency=API_CONCURRENCY,
)
sampling_config = SamplingConfig(
    temperature=0.0, nb_cot_samples=1,
    nb_stepbootstrap_samples=NB_STEPBOOTSTRAP_SAMPLES, seed_stepbootstrap=42,
)
context = SampleContext(model_adapter=adapter, dataset=dataset)
vanilla_sampling = VanillaSampling(generation_config, sampling_config, context)
stepbootstrap_sampling = StepBootstrapSampling(generation_config, sampling_config, context)

print("setup complete")

setup complete


In [5]:
def generate_stepbootstrap_with_progress(sampler):
    """The serial GPT branch of StepBootstrapSampling.generate(), expanded."""
    messages = sampler.context.dataset.build_messages(
        sampler.context.datapoint,
        prompt_request=PromptRequest(few_shot=False, prompt_type=sampler.generation_config.prompt_type),
    )
    alternative_cots = sampler._alternative_cots_stepbootstrap()
    generation_outputs = []

    # Each iteration reaches API_LLM.forward() and makes one OpenAI request.
    for alternative_cot in tqdm(alternative_cots, desc="Step bootstrap API calls", unit="request", leave=False):
        new_messages = sampler._add_assistant_message_to_messages(messages, alternative_cot)
        generate_output = sampler.context.model_adapter.forward_pass(
            messages=new_messages,
            cache=sampler.context.reference_vanilla_question_cache,
        )
        if sampler.generation_config.backend == "api":
            generate_output.answer_token_ids = sampler.context.reference_vanilla_answer_tokens_for_api
        generation_outputs.append(generate_output)

    return generation_outputs

In [8]:
results = []
total_start = perf_counter()

for datapoint in tqdm(datapoints, desc="BigBench Movie", unit="datapoint"):
    context.datapoint = datapoint
    cost_before = adapter.cost()

    start = perf_counter()
    vanilla = vanilla_sampling.generate()
    vanilla_seconds = perf_counter() - start

    context.reference_vanilla_cot = vanilla.cot_steps
    context.reference_vanilla_final_answer = vanilla.final_answer
    context.reference_vanilla_question_cache = vanilla.question_cache
    context.reference_vanilla_answer_tokens_for_api = vanilla.answer_token_ids

    start = perf_counter()
    progress = tqdm(total=NB_STEPBOOTSTRAP_SAMPLES, desc="Step bootstrap API calls", unit="request", leave=False)
    bootstraps = await stepbootstrap_sampling.generate_async(
        concurrency=API_CONCURRENCY,
        progress_callback=progress.update,
    )
    progress.close()
    bootstrap_seconds = perf_counter() - start

    results.append({
        "id": datapoint.id,
        "vanilla_answer": vanilla.final_answer,
        "vanilla_seconds": round(vanilla_seconds, 2),
        "stepbootstrap_samples": len(bootstraps),
        "stepbootstrap_seconds": round(bootstrap_seconds, 2),
        "cost_usd": round(adapter.cost() - cost_before, 6),
    })
    context.clear()

wall_time = perf_counter() - total_start
for result in results:
    print(result)
calls = len(results) * (2 + NB_STEPBOOTSTRAP_SAMPLES)
print(f"\n{calls} API requests across {len(results)} datapoints (step-bootstrap concurrency={API_CONCURRENCY})")
print(f"Total: {wall_time:.2f}s | Mean: {wall_time / len(results):.2f}s/datapoint | Cost: ${sum(r['cost_usd'] for r in results):.6f}")

BigBench Movie:   0%|          | 0/1 [00:00<?, ?datapoint/s]

Step bootstrap API calls:   0%|          | 0/100 [00:00<?, ?request/s]

{'id': 'bigbench_movie_0', 'vanilla_answer': 'C', 'vanilla_seconds': 5.65, 'stepbootstrap_samples': 100, 'stepbootstrap_seconds': 2.21, 'cost_usd': 0.013363}

102 API requests across 1 datapoints (step-bootstrap concurrency=100)
Total: 7.86s | Mean: 7.86s/datapoint | Cost: $0.013363
